In [54]:
import statsmodels.api as sm
import pandas as pd
from patsy import dmatrices
import numpy as np 

Рабочий диапазон амплитуд, определенный экспериментально:
За 1 минуту 32 с амплитуда упала примерно в 1,5 раза. Период колебания приблизительно 4.38 с (21 колебание)

## Расчет параметров устновки и их ошибок

In [55]:
# измерение параметров установки
l = 216*10**(-2) # m
sigma_l = 0.5*10**(-3) # m, половина цены
R = 115.5*10**(-3) # m
sigma_R = 0.5*10**(-3)  # m
r = 30.2*10**(-3) # m
sigma_r = 0.3*10**(-3)  # m

# расчет высоты
z = (l**2 - R**2)**0.5
print(f'z = {z} m')

# замеры для пустой установки
t_0 = 60+32+0.77
n_0 = 21

# расчет k
k = 9.81*R*r/(4*np.pi**2*z)
print(f"k = {k}")

z = 2.1569097686273295 m
k = 0.0004018520798981878


In [56]:
# вывод формул ошибки константы k и расчет ее абсолюнтой ошибки
import sympy as sp
R_sym, r_sym, l_sym, g_sym = sp.symbols('R r l g')
k_func = sp.Function('k')(r_sym, R_sym, l_sym)
k_r = sp.Derivative(k_func, r_sym)
k_R = sp.Derivative(k_func, R_sym)
k_l = sp.Derivative(k_func, l_sym)

k_sym = g_sym*R_sym*r_sym/(4*sp.pi**2*sp.sqrt(l_sym**2 - R_sym**2))
dk_dr = sp.Derivative(k_sym, r_sym, evaluate=True)
dk_dR = sp.Derivative(k_sym, R_sym, evaluate=True)
dk_dl = sp.Derivative(k_sym, l_sym, evaluate=True)

k_eq_1 = sp.Eq(k_r, dk_dr)
k_eq_2 = sp.Eq(k_R, dk_dR)
k_eq_3 = sp.Eq(k_l, dk_dl)

display(k_eq_1, k_eq_2, k_eq_3)

delta_r = sp.symbols(r'\Delta{r}')
delta_R = sp.symbols(r'\Delta{R}')
delta_l = sp.symbols(r'\Delta{l}')
sigma_k = sp.symbols(r'\sigma_k')

term1 = sp.Pow(dk_dr * delta_r, 2, evaluate=False)
term2 = sp.Pow(dk_dR * delta_R, 2, evaluate=False)
term3 = sp.Pow(dk_dl * delta_l, 2, evaluate=False)

result_sigma = sp.Eq(sigma_k, sp.sqrt(term1 + term2 + term3))

display(result_sigma)

display(k_sym)




Eq(Derivative(k(r, R, l), r), R*g/(4*pi**2*sqrt(-R**2 + l**2)))

Eq(Derivative(k(r, R, l), R), R**2*g*r/(4*pi**2*(-R**2 + l**2)**(3/2)) + g*r/(4*pi**2*sqrt(-R**2 + l**2)))

Eq(Derivative(k(r, R, l), l), -R*g*l*r/(4*pi**2*(-R**2 + l**2)**(3/2)))

Eq(\sigma_k, sqrt((\Delta{R}*(R**2*g*r/(4*pi**2*(-R**2 + l**2)**(3/2)) + g*r/(4*pi**2*sqrt(-R**2 + l**2))))**2 + (R*\Delta{r}*g/(4*pi**2*sqrt(-R**2 + l**2)))**2 + (-R*\Delta{l}*g*l*r/(4*pi**2*(-R**2 + l**2)**(3/2)))**2))

R*g*r/(4*pi**2*sqrt(-R**2 + l**2))

In [57]:
subst_dict = {R_sym : R, g_sym: 9.81, r_sym : r, l_sym : l, sp.pi : np.pi, delta_r : sigma_r, delta_R : sigma_R, delta_l : sigma_l}
sigma_k = (result_sigma.subs(subst_dict))
sigma_k

Eq(\sigma_k, 4.35748652772736e-6)

## Измерение момента инерции тел

In [58]:
I_sym, m_sym, T_sym= sp.symbols('I m T')
k_sym_2 = sp.symbols('k')
moment_I = sp.Eq(I_sym, k_c*m_sym*T_sym**2)
display(moment_I)



Eq(I, T**2*k*m)

### Пустая платформа

In [59]:
# момент инерции пустой установки
# замеры для пустой установки
t_0 = 60+32+0.77
n_0 = 21
T_0 = t_0/n_0
m_0 = 1026.4*10**(-3) # kg
sigma_m_0 = 0.5*10**(-3) # kg

platform_moment_dict = {k_sym_2 : k, m_sym : m, T_sym : T}
I_platform = moment_I.subs(platform_moment_dict)
I_platform



Eq(I, 0.00804932360497363)

In [60]:
# расчет ошибки для момента инерции платформы
sigma_t_0 = 0.01  # последняя значщая цифра счетчика
sigma_T_0 = sigma_t_0/n_0

T_0_sym, m_0_sym = sp.symbols('T_0 m_0')
sigma_T_0_sym, sigma_m_0_sym = sp.symbols(r'\sigma_{T_0} \sigma_{m_0}')
I_0_sym = sp.Symbol('I_0')
I_0_expr = k_c*m_0_sym*T_0_sym**2
I_platform = I_0_expr.subs({k_c:k, m_0_sym:m, T_0_sym:T_0})
display(sp.Eq(T_0_sym, sp.Symbol('t_0')/sp.Symbol('n_0')))
display(sp.Eq(T_0_sym, sp.Float(T_0, 7)))
display(sp.Eq(I_0_sym, I_0_expr))
display(sp.Eq(I_0_sym, sp.Float(I_platform, 7)))

Eq(T_0, t_0/n_0)

Eq(T_0, 4.417619)

Eq(I_0, T_0**2*k*m_0)

Eq(I_0, 0.008049324)

In [61]:
# ошибка момента инерции пустой платформы
from IPython.display import Math
sigma_I_0_sym = sp.Symbol(r'\sigma_{I_0}')
vars_platform = [(k_c, sigma_k_sym), (m_0_sym, sigma_m_0_sym),
                 (T_0_sym, sigma_T_0_sym)]
for var, _ in vars_platform:
    display(sp.Eq(sp.Derivative(sp.Function('I_0')(k_c,m_0_sym,T_0_sym), var),
                  sp.diff(I_0_expr, var)))
sigma_I_0_expr = sp.sqrt(sum((sp.diff(I_0_expr, var)*unc)**2
                                for var, unc in vars_platform))
display(sp.Eq(sigma_I_0_sym, sigma_I_0_expr))
platform_subs = {k_c : k, sigma_k_sym : 4.35748652772736*10**(-6), m_0_sym : m_0,
                 sigma_m_0_sym : sigma_m_0, T_0_sym : T_0, sigma_T_0_sym : sigma_T_0}
sigma_I_platform = sigma_I_0_expr.subs(platform_subs)
display(sp.Eq(sigma_I_0_sym, sigma_I_platform))

Eq(Derivative(I_0(k, m_0, T_0), k), T_0**2*m_0)

Eq(Derivative(I_0(k, m_0, T_0), m_0), T_0**2*k)

Eq(Derivative(I_0(k, m_0, T_0), T_0), 2*T_0*k*m_0)

Eq(\sigma_{I_0}, sqrt(T_0**4*\sigma_{m_0}**2*k**2 + T_0**4*m_0**2*sigma_k**2 + 4*T_0**2*\sigma_{T_0}**2*k**2*m_0**2))

Eq(\sigma_{I_0}, 8.73881755020746e-5)

In [62]:
# тело 1 (круг):
d_1 = 170.5*10**(-3)  # м  
m_1 = 587.9*10**(-3)  # кг
sigma_m_1 = 0.5*10**(-3)  # кг
r_1 = d_1/2

moment_1 = pd.DataFrame({'t, c': [60+58+0.96, 60+38+0.8, 60+19+0.19],
                         'n': [30,25,20]})
moment_1['T, c'] = moment_1['t, c']/moment_1['n']
display(moment_1)
T_1 = float(moment_1['T, c'].mean())


,"t, c",n,"T, c"
0,118.96,30,3.965333
1,98.80,25,3.952000
2,79.19,20,3.959500


In [63]:
# измерение момента инерции тела 2 (брусок)
d_2 = 209.4 # mm, длина
w_2 = 28 # mm, ширина
h_2 = 27.6 # mm, высота
m_2 = 1273 # кг
moment_2 = pd.DataFrame({"t, c" : [60+52+0.54, 60+33+0.55, 60+14+0.99], "n" : [30, 25, 20]})
moment_2

,"t, c",n
0,112.54,30
1,93.55,25
2,74.99,20


In [64]:
# измерение момента инерции двух тел 1 и 2 вместе
moment_1_and_2 = pd.DataFrame({'t, с' : [60+47+0.56, 60+29+0.64, 60+12], "n" : [30, 25, 20]})
moment_1_and_2
moment_1_and_2.to_latex(escape=True)


'\\begin{tabular}{lrr}\n\\toprule\n & t, с & n \\\\\n\\midrule\n0 & 107.560000 & 30 \\\\\n1 & 89.640000 & 25 \\\\\n2 & 72.000000 & 20 \\\\\n\\bottomrule\n\\end{tabular}\n'

In [65]:
# измерение момента инерции разрезанного диска
m_1 = 763.5 # г
m_2 = 764 # г
d_disk = 90.9 # mm, диаметр
w_disk = 27.9 # mm, толщина
# h - расстояние от центра половинки до центра платформы в миллиметрах
moment_disk = pd.DataFrame({'h': [0, 5, 15, 30, 40, 55, 65], 't, с' : [60+17+0.25, 60+17+0.06, 60+18+0.34, 60+22+0.46, 60+26+0.02, 60+33+0.36, 60+39+0.02], "n" : [25, 25, 25, 25, 25, 25, 25]})
moment_disk

,h,"t, с",n
0,0,77.25,25
1,5,77.06,25
2,15,78.34,25
3,30,82.46,25
4,40,86.02,25
5,55,93.36,25
6,65,99.02,25
